# ARC-AGI Solver — Kaggle Submission

Generates a Kaggle submission JSON from a trained checkpoint.

For each evaluation task:
- **Attempt 1**: rule-based solver (if applicable), otherwise TTA
- **Attempt 2**: TTA with a different RNG seed (independent prediction — maximises
  chance that at least one attempt is correct)

The evaluation dataset includes ground-truth outputs, so accuracy is also printed.

**Cell order:** 1 → 2 → 3 → 4 → 5

In [ ]:
# ── Cell 1: Check GPU ───────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — inference will be slow on CPU.')

In [ ]:
# ── Cell 2: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_BASE     = '/content/drive/MyDrive/arc-agi'
DRIVE_CKPT_DIR = f'{DRIVE_BASE}/checkpoints'

print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')
ckpts = sorted(Path(DRIVE_CKPT_DIR).glob('*.pt'))
print(f'Available checkpoints ({len(ckpts)}):')
for p in ckpts:
    print(f'  {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

In [ ]:
# ── Cell 3: Clone repo and download ARC data ────────────────────────────────
import os, io, subprocess, urllib.request, zipfile
from pathlib import Path

# Use absolute path so this cell is safe to re-run at any time
REPO        = 'arc-agi-solver'
GITHUB_USER = 'rodehyde'
REPO_DIR    = f'/content/{REPO}'

if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                   check=True)
else:
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

log = subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-3'],
                     capture_output=True, text=True)
print(log.stdout)

# Download ARC-AGI dataset (both training and evaluation)
DATA_DIR  = Path('data')
eval_dir  = DATA_DIR / 'evaluation'
train_dir = DATA_DIR / 'training'

need_download = (
    not eval_dir.exists()  or len(list(eval_dir.glob('*.json')))  < 400 or
    not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400
)

if need_download:
    print('Downloading ARC-AGI dataset...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip'
    ) as r:
        raw = r.read()
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for split in ('training', 'evaluation'):
            dest = DATA_DIR / split
            dest.mkdir(exist_ok=True)
            for m in zf.namelist():
                if f'data/{split}/' in m and m.endswith('.json'):
                    (dest / Path(m).name).write_bytes(zf.read(m))
    print(f"training: {len(list(train_dir.glob('*.json')))} tasks")
    print(f"evaluation: {len(list(eval_dir.glob('*.json')))} tasks")
else:
    print(f"ARC data already present  "
          f"(training: {len(list(train_dir.glob('*.json')))}  "
          f"evaluation: {len(list(eval_dir.glob('*.json')))}")

In [ ]:
# ── Cell 4: Configuration ────────────────────────────────────────────────────
from pathlib import Path

RUN_NAME  = 'all_400_arc'
K_CONTEXT = 3

# ── TTA settings ──────────────────────────────────────────────────────────────
# Attempt 1 uses seed=SEED1; attempt 2 uses seed=SEED2.
# Both use the same N_PERMS and N_D4.
N_PERMS = 20   # colour permutations per D4 orientation
N_D4    = 8    # D4 orientations: 1=colour-only, 8=all D4 symmetries
SEED1   = 42   # RNG seed for attempt 1
SEED2   = 137  # RNG seed for attempt 2

# ── Restrict to specific task IDs (set None to run all 400 evaluation tasks) ──
PILOT_TASK_IDS = None
# PILOT_TASK_IDS = ['00576224', '009d5c81', '00dbd492']  # ← example subset

# ── Resolve checkpoint path ───────────────────────────────────────────────────
ckpt_path = str(Path(DRIVE_CKPT_DIR) / f'transformer_{RUN_NAME}_best.pt')
if Path(ckpt_path).exists():
    sz = Path(ckpt_path).stat().st_size / 1e6
    print(f'Checkpoint: {ckpt_path}  ({sz:.1f} MB)  ✓')
else:
    print(f'Checkpoint NOT FOUND: {ckpt_path}')
    for p in sorted(Path(DRIVE_CKPT_DIR).glob('*.pt')):
        print(f'  {p.name}')

n_tasks = len(PILOT_TASK_IDS) if PILOT_TASK_IDS else 'all 400'
print(f'Tasks: {n_tasks}  |  n_perms={N_PERMS}  n_d4={N_D4}  k_ctx={K_CONTEXT}')

In [ ]:
# ── Cell 5: Generate submission ──────────────────────────────────────────────
import subprocess, sys, shutil
from pathlib import Path

drive_results  = Path(DRIVE_BASE) / 'results'
local_results  = Path('results')
local_results.mkdir(exist_ok=True)
drive_results.mkdir(parents=True, exist_ok=True)

ckpt_stem      = Path(ckpt_path).stem
submission_out = str(local_results / f'submission_{ckpt_stem}.json')

cmd = [
    sys.executable, '-u', 'scripts/submit.py',
    '--checkpoint', ckpt_path,
    '--data-dir',   'data/evaluation',
    '--n-perms',    str(N_PERMS),
    '--n-d4',       str(N_D4),
    '--k-context',  str(K_CONTEXT),
    '--seed',       str(SEED1),
    '--seed2',      str(SEED2),
    '--output-file', submission_out,
    '--verbose',
    *([ '--task-ids'] + PILOT_TASK_IDS if PILOT_TASK_IDS else []),
]

print(f'Tasks: {n_tasks}  |  n_perms={N_PERMS}  n_d4={N_D4}')
print('=' * 60)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'Exit code: {proc.returncode}')

# ── Copy submission JSON to Drive ──────────────────────────────────────────
if proc.returncode == 0 and Path(submission_out).exists():
    dst = drive_results / Path(submission_out).name
    shutil.copy(submission_out, dst)
    sz = Path(submission_out).stat().st_size / 1e3
    print(f'\nSubmission saved to Drive: {dst.name}  ({sz:.0f} KB)')
    print('Download it from Drive and upload to Kaggle.')